In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
import xarray as xr

from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
    brier_score_loss
)


PROJECT = Path(r"Z:\Projects\monsoon-postprocessing")

DATA_FILE = (
    PROJECT
    / "data"
    / "processed"
    / "july2018_model_dataset.nc"
)

OUTPUT_FILE = (
    PROJECT
    / "data"
    / "processed"
    / "heavy_rain_probability_test.nc"
)

METRICS_FILE = (
    PROJECT
    / "data"
    / "processed"
    / "heavy_rain_probability_metrics.csv"
)


with xr.open_dataset(DATA_FILE) as ds:
    model_ds = ds.load()

In [2]:
FEATURES = [
    "raw_gefs",
    "local_mean",
    "local_max",
    "mslp",
    "u850",
    "v850",
    "q850",
    "wind_speed",
    "latitude",
    "longitude"
]


def build_features(dataset):
    raw = dataset["gefs_rainfall"]

    local_mean = raw.rolling(
        latitude=3,
        longitude=3,
        center=True,
        min_periods=1
    ).mean()

    local_max = raw.rolling(
        latitude=3,
        longitude=3,
        center=True,
        min_periods=1
    ).max()

    latitude, longitude = xr.broadcast(
        raw.latitude,
        raw.longitude
    )

    latitude = latitude.broadcast_like(raw)
    longitude = longitude.broadcast_like(raw)

    feature_values = np.column_stack([
        raw.values.ravel(),
        local_mean.values.ravel(),
        local_max.values.ravel(),
        dataset["mslp"].values.ravel(),
        dataset["u850"].values.ravel(),
        dataset["v850"].values.ravel(),
        dataset["q850"].values.ravel(),
        dataset["wind_speed_850"].values.ravel(),
        latitude.values.ravel(),
        longitude.values.ravel()
    ])

    observed = dataset[
        "imerg_rainfall"
    ].values.ravel()

    valid = (
        np.all(
            np.isfinite(feature_values),
            axis=1
        )
        & np.isfinite(observed)
    )

    return (
        feature_values[valid],
        observed[valid],
        valid,
        raw
    )

In [3]:
train_ds = model_ds.sel(
    date=slice("2018-07-01", "2018-07-21")
)

validation_ds = model_ds.sel(
    date=slice("2018-07-22", "2018-07-26")
)

test_ds = model_ds.sel(
    date=slice("2018-07-27", "2018-07-31")
)


X_train, rain_train, _, _ = build_features(
    train_ds
)

X_validation, rain_validation, _, _ = build_features(
    validation_ds
)

X_test, rain_test, test_mask, test_template = (
    build_features(test_ds)
)

In [4]:
def event_metrics(
    probabilities,
    observations,
    rainfall_threshold,
    probability_threshold
):
    observed_event = (
        observations >= rainfall_threshold
    )

    forecast_event = (
        probabilities >= probability_threshold
    )

    hits = np.sum(
        forecast_event & observed_event
    )

    misses = np.sum(
        ~forecast_event & observed_event
    )

    false_alarms = np.sum(
        forecast_event & ~observed_event
    )

    total = len(observations)

    csi = (
        hits / (hits + misses + false_alarms)
        if hits + misses + false_alarms > 0
        else np.nan
    )

    pod = (
        hits / (hits + misses)
        if hits + misses > 0
        else np.nan
    )

    far = (
        false_alarms / (hits + false_alarms)
        if hits + false_alarms > 0
        else np.nan
    )

    random_hits = (
        ((hits + misses) * (hits + false_alarms))
        / total
    )

    ets_denominator = (
        hits + misses + false_alarms
        - random_hits
    )

    ets = (
        (hits - random_hits) / ets_denominator
        if ets_denominator > 0
        else np.nan
    )

    return {
        "hits": int(hits),
        "misses": int(misses),
        "false_alarms": int(false_alarms),
        "CSI": csi,
        "POD": pod,
        "FAR": far,
        "ETS": ets
    }

In [5]:
def select_probability_threshold(
    probabilities,
    observations,
    rainfall_threshold
):
    best_threshold = 0.5
    best_csi = -1

    for threshold in np.linspace(
        0.01,
        0.80,
        80
    ):
        result = event_metrics(
            probabilities,
            observations,
            rainfall_threshold,
            threshold
        )

        if result["CSI"] > best_csi:
            best_csi = result["CSI"]
            best_threshold = threshold

    return best_threshold

In [6]:
RAINFALL_THRESHOLDS = {
    "heavy": 64.5,
    "very_heavy": 115.6
}

metric_rows = []
probability_maps = {}


for event_name, rainfall_threshold in (
    RAINFALL_THRESHOLDS.items()
):
    y_train = (
        rain_train >= rainfall_threshold
    ).astype(int)

    y_validation = (
        rain_validation >= rainfall_threshold
    ).astype(int)

    y_test = (
        rain_test >= rainfall_threshold
    ).astype(int)

    model = HistGradientBoostingClassifier(
        learning_rate=0.08,
        max_iter=150,
        max_leaf_nodes=31,
        min_samples_leaf=40,
        l2_regularization=1.0,
        random_state=42
    )

    model.fit(
        X_train,
        y_train
    )

    validation_probability = (
        model.predict_proba(
            X_validation
        )[:, 1]
    )

    selected_threshold = (
        select_probability_threshold(
            validation_probability,
            rain_validation,
            rainfall_threshold
        )
    )

    # Refit using training and validation data.
    X_train_final = np.vstack([
        X_train,
        X_validation
    ])

    y_train_final = np.concatenate([
        y_train,
        y_validation
    ])

    final_model = HistGradientBoostingClassifier(
        learning_rate=0.08,
        max_iter=150,
        max_leaf_nodes=31,
        min_samples_leaf=40,
        l2_regularization=1.0,
        random_state=42
    )

    final_model.fit(
        X_train_final,
        y_train_final
    )

    test_probability = (
        final_model.predict_proba(
            X_test
        )[:, 1]
    )

    categorical = event_metrics(
        test_probability,
        rain_test,
        rainfall_threshold,
        selected_threshold
    )

    metric_rows.append({
        "event": event_name,
        "rainfall_threshold_mm": rainfall_threshold,
        "probability_threshold": selected_threshold,
        "Brier_score": brier_score_loss(
            y_test,
            test_probability
        ),
        "ROC_AUC": roc_auc_score(
            y_test,
            test_probability
        ),
        "PR_AUC": average_precision_score(
            y_test,
            test_probability
        ),
        **categorical
    })

    full_values = np.full(
        test_template.size,
        np.nan,
        dtype=np.float32
    )

    full_values[test_mask] = (
        test_probability
    )

    probability_maps[event_name] = xr.DataArray(
        full_values.reshape(
            test_template.shape
        ),
        coords=test_template.coords,
        dims=test_template.dims
    )

In [7]:
probability_metrics_df = pd.DataFrame(
    metric_rows
)

probability_metrics_df

,event,rainfall_threshold_mm,probability_threshold,Brier_score,ROC_AUC,PR_AUC,hits,misses,false_alarms,CSI,POD,FAR,ETS
0,heavy,64.5,0.10,0.013978,0.909883,0.116839,234,894,1348,0.094507,0.207447,0.852086,0.085788
1,very_heavy,115.6,0.05,0.002035,0.921333,0.030430,12,133,247,0.030612,0.082759,0.953668,0.029382


In [8]:
probability_ds = xr.Dataset({
    "heavy_rain_probability": (
        probability_maps["heavy"]
    ),
    "very_heavy_rain_probability": (
        probability_maps["very_heavy"]
    ),
    "observed_rainfall": (
        test_ds["imerg_rainfall"]
    )
})

probability_ds[
    "heavy_rain_probability"
].attrs["units"] = "probability"

probability_ds[
    "very_heavy_rain_probability"
].attrs["units"] = "probability"

probability_ds.attrs = {
    "test_period": "2018-07-27 to 2018-07-31",
    "heavy_threshold_mm": 64.5,
    "very_heavy_threshold_mm": 115.6
}

probability_ds.to_netcdf(
    OUTPUT_FILE,
    mode="w",
    engine="netcdf4"
)

probability_metrics_df.to_csv(
    METRICS_FILE,
    index=False
)

print("Probability file saved:", OUTPUT_FILE.exists())
print("Metrics saved:", METRICS_FILE.exists())

Probability file saved: True
Metrics saved: True


In [10]:
from pathlib import Path

import numpy as np
import pandas as pd
import xarray as xr
from sklearn.metrics import brier_score_loss

PROJECT_ROOT = Path(r"Z:\Projects\monsoon-postprocessing")
MODEL_FILE = PROJECT_ROOT / "data" / "processed" / "july2018_model_dataset.nc"
PROBABILITY_FILE = (
    PROJECT_ROOT / "data" / "processed" / "heavy_rain_probability_test.nc"
)

with xr.open_dataset(MODEL_FILE) as dataset:
    rainfall = dataset["imerg_rainfall"].load()

with xr.open_dataset(PROBABILITY_FILE) as dataset:
    probability_ds = dataset.load()

# Training and validation data define climatological event probabilities.
train_validation_rain = rainfall.sel(
    date=slice("2018-07-01", "2018-07-26")
)

test_rain = rainfall.sel(
    date=slice("2018-07-27", "2018-07-31")
)

event_settings = {
    "heavy": {
        "threshold": 64.5,
        "probability_variable": "heavy_rain_probability",
    },
    "very_heavy": {
        "threshold": 115.6,
        "probability_variable": "very_heavy_rain_probability",
    },
}

brier_results = []

for event_name, settings in event_settings.items():
    threshold = settings["threshold"]
    variable = settings["probability_variable"]

    climatology = float(
        (train_validation_rain >= threshold)
        .where(train_validation_rain.notnull())
        .mean(skipna=True)
    )

    probability, observation = xr.align(
        probability_ds[variable],
        test_rain,
        join="inner",
    )

    valid = probability.notnull() & observation.notnull()

    y_probability = probability.where(valid).values.ravel()
    y_observation = (
        observation.where(valid).values.ravel() >= threshold
    ).astype(float)

    valid_flat = np.isfinite(y_probability) & np.isfinite(
        observation.where(valid).values.ravel()
    )

    y_probability = y_probability[valid_flat]
    y_observation = y_observation[valid_flat]

    model_brier = brier_score_loss(
        y_observation,
        y_probability,
    )

    climatology_probability = np.full(
        y_observation.shape,
        climatology,
    )

    climatology_brier = brier_score_loss(
        y_observation,
        climatology_probability,
    )

    brier_skill_score = 1 - (
        model_brier / climatology_brier
    )

    brier_results.append(
        {
            "event": event_name,
            "threshold_mm": threshold,
            "training_climatology": climatology,
            "model_brier": model_brier,
            "climatology_brier": climatology_brier,
            "brier_skill_score": brier_skill_score,
        }
    )

brier_skill_table = pd.DataFrame(brier_results)
brier_skill_table

,event,threshold_mm,training_climatology,model_brier,climatology_brier,brier_skill_score
0,heavy,64.5,0.019100,0.013978,0.014722,0.050551
1,very_heavy,115.6,0.003403,0.002035,0.001917,-0.061347


In [11]:
print(list(probability_ds.data_vars))

['heavy_rain_probability', 'very_heavy_rain_probability', 'observed_rainfall']
